# 🚀 요미(Yomi) 3B 모델 QLoRA 파인튜닝 (구글 코랩용)

이 노트북은 **Unsloth** 라이브러리를 사용하여 빠르고 효율적으로 요미 페르소나 모델을 파인튜닝합니다.
수면 중 런타임 끊김 방지를 위해 브라우저 개발자 도구(F12) -> Console 창에 아래 코드를 입력해두는 것을 권장합니다.

```javascript
function ClickConnect(){
    console.log("Working"); 
    document.querySelector("colab-connect-button").click() 
}
setInterval(ClickConnect, 60000)
```

In [ ]:
# 1. 구글 드라이브 마운트 (체크포인트 저장 및 데이터셋 로드용)
from google.colab import drive
drive.mount('/content/drive')

### 🛑 주의사항
`yomi_dataset.jsonl` 파일을 구글 드라이브의 `MyDrive/Yomi_Project/` 폴더에 미리 업로드해두어야 합니다!

In [ ]:
# 2. Unsloth 및 종속성 설치 (최초 1회 실행 시 약 2~3분 소요)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 3. 모델 및 토크나이저 로드 (3B 모델 기준)
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # 모바일 환경을 고려해 너무 길지 않게 설정
dtype = None
load_in_4bit = True # 4bit 양자화로 메모리 절약

# 베이스 모델: Qwen2.5-3B-Instruct (또는 0.5B 사용시 Qwen/Qwen2.5-0.5B-Instruct로 변경)
model_name = "Qwen/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# LoRA 어댑터 설정
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank 사이즈
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout = 0 is optimized for Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 4. 데이터셋 로드 및 포맷팅
from datasets import load_dataset

# 구글 드라이브에 업로드한 데이터셋 경로
data_path = "/content/drive/MyDrive/Yomi_Project/yomi_dataset.jsonl"

# 데이터셋 로드
dataset = load_dataset("json", data_files=data_path, split="train")

# Qwen Chat Template 적용 함수
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Total dataset size: {len(dataset)}")

In [ ]:
# 5. 훈련 설정 (Trainer)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

output_dir = "/content/drive/MyDrive/Yomi_Project/checkpoints"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        max_steps = 1500, # 약 1에폭 정도의 스텝 (필요시 조정)
        # num_train_epochs = 1, # max_steps 대신 사용 가능
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = output_dir,
        save_strategy = "steps",
        save_steps = 250, # 250 스텝마다 드라이브에 안전하게 자동 저장!
    ),
)

In [ ]:
# 6. 파인튜닝 시작! (자러 가셔도 됩니다 😴)
# 만약 중간에 런타임이 끊겼다면 resume_from_checkpoint = True 옵션을 켜면 이어서 학습합니다.
trainer_stats = trainer.train(resume_from_checkpoint = False)

In [ ]:
# 7. 학습 완료된 모델을 GGUF(모바일 유니티용)로 변환하여 구글 드라이브에 저장
import os
gguf_save_path = "/content/drive/MyDrive/Yomi_Project/yomi_3b_q4_k_m"
os.makedirs(gguf_save_path, exist_ok=True)

model.save_pretrained_gguf(gguf_save_path, tokenizer, quantization_method = "q4_k_m")
print("✨ GGUF 저장 완료! 유니티에 탑재할 준비가 되었습니다 ✨")